# Environment Setup

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

## Data Processsing

In [ ]:
# Load the uploaded CSV file
file_path = '/Users/xanderyoon/Downloads/EDGARv8.0_total_GHG_GWP100_AR5_NUTS2_1990_2022.csv'
df = pd.read_csv(file_path, encoding='ISO-8859-1', skiprows=5)

# Melt the dataframe to transform years into a target variable
df = df.melt(id_vars=['NUTS 2'], value_vars=df.columns[5:],
                            var_name='Year', value_name='Value')

# Clean the 'Year' column to extract just the year number
df['Year'] = df['Year'].str.extract('(\d+)')
df = df.dropna()
df.head()

,NUTS 2,Year,Value
0,AT,1990,405.240107
1,AT11,1990,1998.728735
2,AT12,1990,20728.716120
3,AT13,1990,4704.881300
4,AT21,1990,5264.441540


In [ ]:
# Split into train, validation, and test sets
df_train = df[df['Year'].astype(int).between(1990, 2019)]
df_test = df[df['Year'].astype(int).between(2020, 2022)]

# Split into X, y variables
X_train = df_train[['NUTS 2','Year']]
y_train = df_train['Value']
X_test = df_test[['NUTS 2', 'Year']]
y_test = df_test['Value']

In [ ]:
# Define preprocessing: One-Hot Encoding for 'NUTS 2' and scaling for all features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Year']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['NUTS 2'])
    ])

# Decision Tree

A **Decision Tree** is a supervised learning algorithm used for classification and regression tasks. It works by splitting the dataset into subsets based on the value of input features, forming a tree-like structure where each internal node represents a feature (or attribute), each branch represents a decision rule, and each leaf node represents an outcome or prediction.

## TL;DR

**What it does**: A decision tree splits data into branches based on feature values to make predictions, forming a tree-like model that is easy to interpret. It works well for both classification and regression tasks.

**When to use it**: Use decision trees when you need a model that is simple to understand and interpret, can handle both numerical and categorical data, and does not require feature scaling. They are suitable for smaller datasets and can handle non-linear relationships.

## How It Works

1. **Splitting**: The dataset is split based on the feature that results in the most significant information gain or reduction in impurity.
2. **Stopping Criteria**: The splitting continues recursively until a stopping criterion is met, such as a maximum depth or a minimum number of samples per leaf.
3. **Prediction**: For classification, the prediction is the mode (most frequent class) of the training samples in the leaf node. For regression, it is the mean value of the samples in the leaf.

## Mathematical Concepts

### 1. **Entropy and Information Gain**

Entropy measures the impurity or uncertainty in a dataset. For a binary classification problem, the entropy $ H $ of a node is given by:

$ H(X) = - p_1 \log_2(p_1) - p_2 \log_2(p_2) $

where $ p_1 $ and $ p_2 $ are the probabilities of the two classes.

**Information Gain (IG)** is used to decide which feature to split on at each step in the tree. It is calculated as:

$
IG(T, X) = H(T) - \sum_{i=1}^{n} \frac{|T_i|}{|T|} H(T_i)
$

where:
- $ T $ is the original dataset,
- $ T_i $ are the subsets of $ T $ after the split on feature $ X $,
- $ H(T) $ is the entropy of the original dataset,
- $ H(T_i) $ is the entropy of subset $ T_i $,
- $ |T| $ and $ |T_i| $ are the sizes of the dataset and subset respectively.

### 2. **Reduction in Variance (for Regression)**

For regression tasks, the decision tree uses Reduction in Variance to decide the splits. The variance is given by:

$
Var(T) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \mu)^2
$

where:
- $ n $ is the number of instances,
- $ y_i $ are the target values,
- $ \mu $ is the mean of the target values in the node.

The split is chosen to minimize the variance within the subsets formed by the split.

Decision Trees are prone to overfitting, so techniques like pruning, setting a maximum depth, or a minimum number of samples per leaf are used to control the tree's complexity.


In [ ]:
# Define the pipeline: Preprocessing followed by the Decision Tree Regressor
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=42))
])

# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

# Make predictions on the test set
y_pred = pipeline.predict(X_test)

# Evaluate the model using R-squared
r2 = r2_score(y_test, y_pred)
print(f'R-squared: {r2}')

R-squared: 0.9850863755930146


# Bagging

**Bagging** (Bootstrap Aggregating) is an ensemble learning method that aims to improve the stability and accuracy of machine learning algorithms. It reduces variance and helps prevent overfitting by combining the predictions of multiple models (often decision trees) trained on different subsets of the data. Bagging is commonly used with decision trees, resulting in models like Random Forests.


## TL;DR

**What it does**: Bagging combines the predictions of multiple models trained on randomly sampled subsets of the data to create a more robust and accurate model. It reduces the model's variance and helps prevent overfitting.

**When to use it**: Use bagging when you want to improve the performance of models that have high variance, such as decision trees, especially when dealing with noisy data or datasets with high variability. It's particularly effective when the base model is prone to overfitting.

## How Bagging Works

1. **Bootstrap Sampling**: From the original dataset, multiple subsets are created using bootstrapping (random sampling with replacement). Each subset is of the same size as the original dataset.
   
2. **Training**: A separate model is trained on each subset independently. For example, if using decision trees, each subset is used to train a different tree.

3. **Aggregation**: For classification, the predictions from each model are combined by majority voting. For regression, the predictions are averaged.

## Mathematical Concepts

### 1. **Bootstrap Sampling**

Given a dataset $ D $ of size $ n $, bootstrapping involves sampling $ n $ instances from $ D $ with replacement. This results in $ m $ subsets $ \{D_1, D_2, ..., D_m\} $, where each subset is used to train a separate model.

### 2. **Variance Reduction**

The key idea behind bagging is variance reduction. If $ f(x) $ is the prediction function of a model and each individual model $ f_i(x) $ has an error with variance $ \sigma^2 $, the variance of the aggregated model's prediction (average of all models) reduces to:

$
Var\left(\frac{1}{m} \sum_{i=1}^{m} f_i(x)\right) = \frac{\sigma^2}{m}
$

where $ m $ is the number of models. This shows that as the number of models increases, the variance of the combined prediction decreases.

### 3. **Aggregation (Voting or Averaging)**

For classification:
- The final prediction is determined by majority voting across the $ m $ models:
  
$
\hat{y} = \text{mode}(\hat{y}_1, \hat{y}_2, ..., \hat{y}_m)
$

For regression:
- The final prediction is the average of the predictions from all models:

$
\hat{y} = \frac{1}{m} \sum_{i=1}^{m} \hat{y}_i
$

### 4. **Bias-Variance Tradeoff**

Bagging primarily reduces variance without increasing bias significantly. While individual decision trees may have high variance, combining multiple trees through bagging leads to a more stable model with lower variance, hence improving the overall predictive performance.

Bagging is especially effective when used with models that have high variance but low bias, like decision trees, making it a powerful method for ensemble learning.


In [ ]:
# Define the base model (Decision Tree) within a Bagging Regressor
bagging_model = BaggingRegressor(n_estimators=100, random_state=42)

# Create a pipeline with preprocessing and the bagging model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', bagging_model)
])

# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

# Make predictions on the test set
y_pred = pipeline.predict(X_test)

# Evaluate the model using R^2
r2 = r2_score(y_test, y_pred)
print(f'R-squared: {r2}')

R-squared: 0.9765241860976974


# Boosting

**Boosting** is an ensemble learning technique that aims to create a strong model by combining multiple weak models, typically decision trees, in a sequential manner. Unlike bagging, where models are trained independently, boosting adjusts the weights of incorrectly predicted instances, allowing the next model in the sequence to focus more on the hard-to-predict cases. This approach helps in reducing both bias and variance, leading to improved model performance.

## TL;DR

**What it does**: Boosting combines multiple weak models, like shallow decision trees, into a strong model by sequentially focusing on the mistakes of previous models. It iteratively improves the model's performance by reducing errors.

**When to use it**: Use boosting when you need high accuracy and have sufficient computational resources, especially when working with complex, noisy datasets where a single model might struggle. It is particularly effective for reducing both bias and variance in predictive models.

## How Boosting Works

1. **Initialization**: Start with an initial model that predicts the target variable.
   
2. **Sequential Learning**: Models are added one at a time, and each new model focuses on correcting the errors made by the previous models. This is achieved by adjusting the weights of the incorrectly predicted instances.

3. **Combination**: The predictions of all models are combined, typically through weighted voting (classification) or weighted averaging (regression), to produce the final output.

## Mathematical Concepts

### 1. **Weight Adjustment**

In boosting, each instance in the dataset has an associated weight. Initially, all weights are set equally. After each iteration, weights are updated to emphasize the incorrectly predicted instances, allowing the next model to focus more on these errors.

### 2. **AdaBoost Algorithm**

For AdaBoost, a popular boosting algorithm, the weight update rule is:

$
w_i = w_i \times e^{\alpha \times \text{error}(x_i)}
$

where:
- $ w_i $ is the weight of instance $ i $,
- $ \alpha $ is a factor that depends on the error rate of the model,
- $ \text{error}(x_i) $ indicates whether the prediction was correct or not.

The model's influence ($ \alpha $) is determined by:

$
\alpha = \frac{1}{2} \log\left(\frac{1 - \text{error}}{\text{error}}\right)
$

where `error` is the weighted error rate of the model.

### 3. **Gradient Boosting**

Gradient Boosting builds models sequentially, like AdaBoost, but it minimizes a loss function using gradient descent. It adds a new model that points in the negative gradient direction of the loss function concerning the current ensemble's prediction.

For a regression problem, the update rule can be summarized as:

$
F_{m}(x) = F_{m-1}(x) + \gamma \cdot h_m(x)
$

where:
- $ F_{m-1}(x) $ is the current prediction,
- $ h_m(x) $ is the new weak model trained on the residuals,
- $ \gamma $ is the learning rate that scales the contribution of each model.

### 4. **XGBoost**

XGBoost (Extreme Gradient Boosting) is a powerful implementation of gradient boosting that includes regularization to prevent overfitting, handling missing values, and parallel processing for speed.

Boosting methods like AdaBoost, Gradient Boosting, and XGBoost are highly effective for improving model accuracy, especially when dealing with complex datasets. However, they are more computationally intensive and can be prone to overfitting if not properly tuned.


In [ ]:
# Define the Gradient Boosting Regressor model
gradient_boosting_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)

# Create a pipeline with preprocessing and the gradient boosting model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', gradient_boosting_model)
])

# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

# Make predictions on the test set
y_pred = pipeline.predict(X_test)

# Evaluate the model using R^2
r2 = r2_score(y_test, y_pred)
print(f'R-squared: {r2}')

R-squared: 0.577174590118166


# Random Forest

**Random Forest** is an ensemble learning method that builds multiple decision trees and merges their predictions to improve the accuracy and robustness of the model. It combines the concepts of bagging and random feature selection, making it effective for both classification and regression tasks. By averaging the results of multiple trees, Random Forest reduces overfitting and increases prediction accuracy.

## TL;DR

**What it does**: Random Forest creates multiple decision trees using bootstrapped samples of the data and random subsets of features. It then aggregates their predictions (via majority voting for classification or averaging for regression) to produce a final output that is more accurate and less prone to overfitting than a single tree.

**When to use it**: Use Random Forest when you need a powerful, robust model that can handle large datasets with higher dimensionality, and when overfitting is a concern with individual decision trees. It works well for both classification and regression tasks, especially when interpretability is less critical.

## How Random Forest Works

1. **Bootstrap Sampling**: From the original dataset, multiple subsets are created using bootstrapping (random sampling with replacement). Each subset is used to train a different decision tree.

2. **Random Feature Selection**: At each split in the decision tree, a random subset of features is selected. This ensures that the trees are diverse and reduces correlation among them.

3. **Training**: Each tree is trained independently on its respective subset of data, growing to its full depth without pruning.

4. **Aggregation**: For classification, the final prediction is made by majority voting across all the trees. For regression, the final prediction is the average of all tree predictions.

## Mathematical Concepts

### 1. **Bootstrap Sampling**

Given a dataset $ D $ of size $ n $, bootstrapping involves sampling $ n $ instances from $ D $ with replacement. This results in $ m $ subsets $ \{D_1, D_2, ..., D_m\} $, where each subset is used to train a separate tree.

### 2. **Random Feature Selection**

For each node split in a tree, instead of using all features, a random subset of $ k $ features is selected. The best split is then chosen only from this subset. This randomness helps in creating decorrelated trees, which when aggregated, result in a lower variance model.

### 3. **Aggregation (Voting or Averaging)**

- **Classification**: The final class prediction is determined by majority voting among the trees:
  
$
\hat{y} = \text{mode}(\hat{y}_1, \hat{y}_2, ..., \hat{y}_m)
$

- **Regression**: The final prediction is the average of the predictions from all trees:

$
\hat{y} = \frac{1}{m} \sum_{i=1}^{m} \hat{y}_i
$

### 4. **Out-of-Bag Error (OOB Error)**

Random Forest provides an internal estimate of error, known as Out-of-Bag (OOB) error, by using data that was not included in the bootstrap sample for each tree. This error estimate helps in evaluating the model without the need for a separate validation set.

### 5. **Variance Reduction**

The combination of bootstrap sampling and random feature selection reduces the correlation between individual trees, leading to a significant reduction in variance and an overall robust ensemble model.

Random Forest is a versatile and powerful algorithm that performs well across various types of data, making it a popular choice in both academic research and industry applications.


In [ ]:
random_forest_model = RandomForestRegressor(n_estimators=100, random_state=42)

# Create a pipeline with preprocessing and the random forest model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', random_forest_model)
])

# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

# Make predictions on the test set
y_pred = pipeline.predict(X_test)

# Evaluate the model using R^2
r2 = r2_score(y_test, y_pred)
print(f'R-squared: {r2}')

R-squared: 0.976524192971592


# Ensemble Models

In [ ]:
# Initializing PyCaret setup
reg_setup = setup(data=X_train, target=, session_id=124)

# Comparing all regression models
compare_models()